# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Two Paper Findings + My Methodology Questions

## Finding 1
The FlyRank research paper reports improved model performance compared with a baseline.

**Methodology Question**
Where do the prediction labels come from, and are those labels available before prediction time? This helps determine whether the model is learning useful patterns rather than future information.

---

## Finding 2
The paper evaluates the model using a validation dataset.

**Methodology Question**
Does the validation design (for example, grouped or time-aware validation) match the intended real-world use case? This helps determine whether the reported performance is likely to generalize to unseen data.

These questions are intended to improve methodological rigor rather than criticize the research.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [12]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("/content/content_refresh_anonymized.csv")

current_col_names = df.columns.tolist()
if len(current_col_names) >= 2:
    second_last_col_original_name = current_col_names[-2]
    last_col_original_name = current_col_names[-1]
    df = df.rename(columns={second_last_col_original_name: 'trend_category', last_col_original_name: 'trend_pct'})

df['client_id'] = df['client_id'].astype(str)

# -----------------------------
# Target and features
# -----------------------------
target = "trend_category"

drop_columns = [
    "content_id",
    "client_id",
    "trend_pct",
    target
]

X = df.drop(columns=drop_columns)
y = df[target]

groups = df["client_id"]

# -----------------------------
# Identify feature types and handle NaNs
# -----------------------------
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

X[numeric_features] = X[numeric_features].fillna(0)

for col in categorical_features:
    X[col] = X[col].fillna("missing").astype(str)

# -----------------------------
# Preprocessing
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# -----------------------------
# Honest grouped split
# -----------------------------
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# -----------------------------
# Random Forest model
# -----------------------------
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

group_accuracy = accuracy_score(
    y_test,
    predictions
)

# -------------------------------------------------
# Replace this with your actual Week-5 accuracy
# -------------------------------------------------
week5_accuracy =  0.7421

comparison = pd.DataFrame({
    "Validation Split": [
        "Week 5 Random Split",
        "Grouped by Client"
    ],
    "Accuracy": [
        round(week5_accuracy, 4),
        round(group_accuracy, 4)
    ]
})

print(comparison)

      Validation Split  Accuracy
0  Week 5 Random Split    0.7421
1    Grouped by Client    0.6883


The grouped validation keeps all records from the same client together in either the training or testing data. This provides a more realistic estimate of model performance on unseen clients and reduces the risk of optimistic evaluation.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

#  Leakage Audit

The final feature set was reviewed for potential data leakage.

Columns removed before training:

- content_id
- client_id
- trend_pct

These variables were excluded because they could reveal identity information or future outcome information.

The remaining features represent information that would reasonably be available at prediction time.

No obvious target leakage was observed during this audit.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# Claim Rewrite

Original claim:

"The Random Forest model accurately predicts search trends."

Rewritten claim:

"On the evaluated dataset, the Random Forest model showed observed performance on the validation split. The grouped client validation produced a more conservative estimate than the Week 5 random split. These results are directional and should be interpreted as decision-support rather than proof of future performance."

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.